In [5]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers
import matplotlib.pyplot as plt
import os
import time

NUM_EPOCHS = 2
ABSOLUTE_PATH = os.getcwd()
LAB_PATH = ABSOLUTE_PATH + "/lab06/"

# 1. Завантаження датасету
(train_images, train_labels), (test_images, test_labels) = keras.datasets.cifar10.load_data()

# 2. Ініціалізація назв класів
CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

# 3. Отримання даних для валідації (останні 500 зображень з навчального набору)
validation_images, validation_labels = train_images[:500], train_labels[:500]
train_images, train_labels = train_images[500:], train_labels[500:]

# 4. Перетворення на tf.data.Dataset
train_ds = tf.data.Dataset.from_tensor_slices((train_images, train_labels))
test_ds = tf.data.Dataset.from_tensor_slices((test_images, test_labels))
validation_ds = tf.data.Dataset.from_tensor_slices((validation_images, validation_labels))

# 5. Функція попередньої обробки
def process_images(image, label):
    # Зміна розміру до 227x227 (як очікує оригінальний AlexNet)
    image = tf.image.resize(image, (227, 227))
    # Нормалізація (приведення до діапазону [0, 1])
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

# 6. Створення пайплайну (обробка, перемішування, пакетування)
train_ds = train_ds.map(process_images).shuffle(buffer_size=1000).batch(32)
validation_ds = validation_ds.map(process_images).batch(32)
test_ds = test_ds.map(process_images).batch(32)

# 7. Архітектура AlexNet
model = models.Sequential([
    layers.Conv2D(96, (11, 11), strides=4, activation='relu', input_shape=(227, 227, 3)),
    layers.MaxPooling2D((3, 3), strides=2),
    layers.Conv2D(256, (5, 5), padding='same', activation='relu'),
    layers.MaxPooling2D((3, 3), strides=2),
    layers.Conv2D(384, (3, 3), padding='same', activation='relu'),
    layers.Conv2D(384, (3, 3), padding='same', activation='relu'),
    layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((3, 3), strides=2),
    layers.Flatten(),
    layers.Dense(4096, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(4096, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

# 8. Налаштування TensorBoard
root_logdir = LAB_PATH + "/logs"
def get_run_logdir():
    import time
    run_id = time.strftime("run_%Y_%m_%d-%H_%M_%S")
    return os.path.join(root_logdir, run_id)

run_logdir = get_run_logdir()
tensorboard_cb = keras.callbacks.TensorBoard(run_logdir)

# 9. Компіляція мережі
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Опис мережі
model.summary()

# 10. Навчання
history = model.fit(train_ds, epochs=NUM_EPOCHS, validation_data=validation_ds, callbacks=[tensorboard_cb])

# 11. Оцінка
test_loss, test_acc = model.evaluate(test_ds)
print(f"Точність на тестових даних: {test_acc}")


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_15 (Conv2D)              │ (None, 55, 55, 96)     │        34,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 27, 27, 96)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_16 (Conv2D)              │ (None, 27, 27, 256)    │       614,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 13, 13, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_17 (Conv2D)              │ (None, 13, 13, 384)    │       885,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_18 (Conv2D)              │ (None, 13, 13, 384)    │     1,327,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_19 (Conv2D)              │ (None, 13, 13, 256)    │       884,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 9216)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 4096)           │    37,752,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 4096)           │    16,781,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 10)             │        40,970 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 58,322,314 (222.48 MB)

 Trainable params: 58,322,314 (222.48 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/2
1547/1547 ━━━━━━━━━━━━━━━━━━━━ 4266s 3s/step - accuracy: 0.2285 - loss: 2.1162 - val_accuracy: 0.4300 - val_loss: 1.5285
Epoch 2/2
1547/1547 ━━━━━━━━━━━━━━━━━━━━ 4419s 3s/step - accuracy: 0.4186 - loss: 1.6136 - val_accuracy: 0.4980 - val_loss: 1.3939
313/313 ━━━━━━━━━━━━━━━━━━━━ 206s 659ms/step - accuracy: 0.4990 - loss: 1.4093
Точність на тестових даних: 0.48809999227523804
